# 🚗 BEV-Occ: Bird's-Eye-View 2D Occupancy from Camera Images

**Mobility Hackathon — Round 01 Submission**

This notebook trains and evaluates the BEV-Occ model end-to-end on nuScenes.

**Pipeline:** Multi-camera images → EfficientNet-B4 → Lift-Splat-Shoot → Temporal Fusion → BEV Encoder → Occupancy Grid

**Key Novelties:**
- Distance-weighted BCE loss (directly optimizes competition metric)
- Multi-scale deep supervision
- Temporal BEV fusion via ego-motion-compensated ConvGRU
- 10-sweep LiDAR GT aggregation

> ⚡ **Runtime:** Set to **GPU** (Runtime → Change runtime type → T4 GPU)


## 1. Environment Setup

In [ ]:
# Verify GPU is available
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")


In [ ]:
# Install dependencies
!pip install -q efficientnet_pytorch nuscenes-devkit pyquaternion tensorboard

# Verify
import efficientnet_pytorch
print("✓ All dependencies installed")


## 2. Download nuScenes Mini Dataset

In [ ]:
import os

DATAROOT = "/content/data/nuscenes"
os.makedirs(DATAROOT, exist_ok=True)

# nuScenes v1.0-mini (~4GB)
# You need to get the download link from https://www.nuscenes.org/nuscenes#download
# After registering, you'll get URLs. Paste them below.

# Option A: If you have the tar.gz files uploaded to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/nuscenes/v1.0-mini.tgz /content/data/
# !tar -xzf /content/data/v1.0-mini.tgz -C {DATAROOT}

# Option B: Direct download (replace with your actual URLs from nuscenes.org)
# After registering at nuscenes.org, download these files:
# 1. v1.0-mini metadata
# 2. v1.0-mini file blobs (CAM, LIDAR)

print("="*60)
print("INSTRUCTIONS:")
print("="*60)
print()
print("1. Go to https://www.nuscenes.org/nuscenes#download")
print("2. Register/login and accept the terms")
print("3. Download 'Mini' split (v1.0-mini)")
print("4. Upload the .tgz file to this Colab, OR")
print("   upload to Google Drive and mount it")
print()
print("After downloading, uncomment the appropriate option above")
print("and run this cell again.")
print()
print(f"Expected structure after extraction:")
print(f"  {DATAROOT}/")
print(f"  ├── maps/")
print(f"  ├── samples/")
print(f"  ├── sweeps/")
print(f"  └── v1.0-mini/")

# Check if already downloaded
if os.path.exists(os.path.join(DATAROOT, "v1.0-mini")):
    from nuscenes.nuscenes import NuScenes
    nusc = NuScenes(version="v1.0-mini", dataroot=DATAROOT, verbose=True)
    print(f"\n✓ nuScenes mini loaded: {len(nusc.sample)} samples, {len(nusc.scene)} scenes")
else:
    print(f"\n⏳ Waiting for dataset at {DATAROOT}/v1.0-mini/")


In [ ]:
# Alternative: If you have a direct link or Google Drive file ID
# Uncomment and fill in:

# !pip install -q gdown
# !gdown <your-google-drive-file-id> -O /content/data/v1.0-mini.tgz
# !tar -xzf /content/data/v1.0-mini.tgz -C /content/data/nuscenes/

# Or if using wget with a direct URL:
# !wget -q <direct-url> -O /content/data/v1.0-mini.tgz
# !tar -xzf /content/data/v1.0-mini.tgz -C /content/data/nuscenes/

print("Uncomment the method that works for you, then re-run this cell.")


## 3. Model Code

All model components in a single cell for Colab convenience.
The same code is modularized in the GitHub repo.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
from efficientnet_pytorch import EfficientNet


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BACKBONE: EfficientNet-B4 + FPN Neck + DepthNet
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class EfficientNetBackbone(nn.Module):
    """EfficientNet-B4 with FPN neck. Outputs stride-16 features."""

    def __init__(self, out_channels=160, pretrained=True):
        super().__init__()
        self.out_channels = out_channels
        if pretrained:
            self.backbone = EfficientNet.from_pretrained("efficientnet-b4")
        else:
            self.backbone = EfficientNet.from_name("efficientnet-b4")

        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.lateral_s16 = nn.Conv2d(448, out_channels, 1)
        self.lateral_s8 = nn.Conv2d(160, out_channels, 1)
        self.smooth = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        feats = self.backbone.extract_endpoints(x)
        p_s16 = self.lateral_s16(feats["reduction_5"])
        p_s8 = self.lateral_s8(feats["reduction_4"])
        p_s8 = p_s8 + self.up(p_s16)
        return self.smooth(p_s8)


class DepthNet(nn.Module):
    """Predicts depth distribution + context features per pixel."""

    def __init__(self, in_channels=160, mid_channels=256, depth_channels=112):
        super().__init__()
        self.depth_net = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels), nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, mid_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels), nn.ReLU(inplace=True),
        )
        self.depth_head = nn.Conv2d(mid_channels, depth_channels, 1)
        self.context_head = nn.Conv2d(mid_channels, in_channels, 1)

    def forward(self, features):
        x = self.depth_net(features)
        depth = self.depth_head(x).softmax(dim=1)
        context = self.context_head(x)
        return depth, context


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LIFT-SPLAT: View Transformation
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class LiftSplat(nn.Module):
    """Lift-Splat-Shoot BEV transform."""

    def __init__(self, feat_dim=160, depth_channels=112,
                 x_bound=(-50.0, 50.0, 0.5), y_bound=(-50.0, 50.0, 0.5),
                 z_bound=(-5.0, 3.0, 8.0), depth_min=1.0, depth_max=57.0):
        super().__init__()
        self.feat_dim = feat_dim
        self.depth_channels = depth_channels
        self.depth_min, self.depth_max = depth_min, depth_max
        self.x_bound, self.y_bound, self.z_bound = x_bound, y_bound, z_bound
        self.nx = int((x_bound[1] - x_bound[0]) / x_bound[2])
        self.ny = int((y_bound[1] - y_bound[0]) / y_bound[2])
        self.depth_bins = torch.linspace(depth_min, depth_max, depth_channels)
        self.frustum = None

    def create_frustum(self, feat_h, feat_w, device):
        depth_bins = self.depth_bins.to(device)
        ys = torch.arange(feat_h, device=device).float()
        xs = torch.arange(feat_w, device=device).float()
        ys, xs = torch.meshgrid(ys, xs, indexing="ij")
        ds = depth_bins.view(-1, 1, 1).expand(-1, feat_h, feat_w)
        xs = xs.unsqueeze(0).expand(self.depth_channels, -1, -1)
        ys = ys.unsqueeze(0).expand(self.depth_channels, -1, -1)
        return torch.stack([xs, ys, ds], dim=-1)

    def get_geometry(self, intrinsics, extrinsics, feat_h, feat_w, img_h, img_w):
        device = intrinsics.device
        B, N = intrinsics.shape[:2]
        if self.frustum is None or self.frustum.device != device:
            self.frustum = self.create_frustum(feat_h, feat_w, device)
        frustum = self.frustum.clone()
        frustum[..., 0] *= img_w / feat_w
        frustum[..., 1] *= img_h / feat_h
        D, fH, fW, _ = frustum.shape
        points = frustum.view(-1, 3).clone()
        points[:, 0] *= points[:, 2]
        points[:, 1] *= points[:, 2]
        inv_K = torch.inverse(intrinsics)
        points = points.unsqueeze(0).unsqueeze(0).expand(B, N, -1, -1)
        cam_coords = torch.einsum("bnij,bnpj->bnpi", inv_K, points)
        ones = torch.ones_like(cam_coords[..., :1])
        cam_h = torch.cat([cam_coords, ones], dim=-1)
        ego_coords = torch.einsum("bnij,bnpj->bnpi", extrinsics, cam_h)
        return ego_coords[..., :3].view(B, N, D, fH, fW, 3)

    def voxel_pooling(self, geom, features):
        B, N, D, H, W, C = features.shape
        device = features.device
        x_idx = ((geom[..., 0] - self.x_bound[0]) / self.x_bound[2]).long()
        y_idx = ((geom[..., 1] - self.y_bound[0]) / self.y_bound[2]).long()
        valid = (x_idx >= 0) & (x_idx < self.nx) & (y_idx >= 0) & (y_idx < self.ny)
        x_flat, y_flat = x_idx[valid], y_idx[valid]
        feats_flat = features[valid]
        batch_idx = torch.arange(B, device=device).view(B, 1, 1, 1, 1).expand(B, N, D, H, W)[valid]
        linear_idx = batch_idx * (self.nx * self.ny) + x_flat * self.ny + y_flat
        bev = torch.zeros(B * self.nx * self.ny, C, device=device)
        bev.scatter_add_(0, linear_idx.unsqueeze(-1).expand(-1, C), feats_flat)
        return bev.view(B, self.nx, self.ny, C).permute(0, 3, 1, 2)

    def forward(self, image_features, depth_dist, context_features,
                intrinsics, extrinsics, img_h, img_w):
        B, N, C, H, W = context_features.shape
        D = depth_dist.shape[2]
        geom = self.get_geometry(intrinsics, extrinsics, H, W, img_h, img_w)
        context_perm = context_features.permute(0, 1, 3, 4, 2).unsqueeze(2)
        depth_exp = depth_dist.unsqueeze(-1)
        volume = depth_exp * context_perm
        return self.voxel_pooling(geom, volume)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TEMPORAL FUSION: ConvGRU + Ego-Motion Warping
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class ConvGRUCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        p = kernel_size // 2
        self.reset_gate = nn.Sequential(
            nn.Conv2d(input_dim + hidden_dim, hidden_dim, kernel_size, padding=p, bias=False),
            nn.BatchNorm2d(hidden_dim), nn.Sigmoid())
        self.update_gate = nn.Sequential(
            nn.Conv2d(input_dim + hidden_dim, hidden_dim, kernel_size, padding=p, bias=False),
            nn.BatchNorm2d(hidden_dim), nn.Sigmoid())
        self.candidate = nn.Sequential(
            nn.Conv2d(input_dim + hidden_dim, hidden_dim, kernel_size, padding=p, bias=False),
            nn.BatchNorm2d(hidden_dim), nn.Tanh())

    def forward(self, x, h):
        combined = torch.cat([x, h], dim=1)
        r = self.reset_gate(combined)
        z = self.update_gate(combined)
        h_cand = self.candidate(torch.cat([x, r * h], dim=1))
        return (1 - z) * h + z * h_cand


class TemporalFusion(nn.Module):
    def __init__(self, bev_dim=64, hidden_dim=64,
                 x_bound=(-50.0, 50.0, 0.5), y_bound=(-50.0, 50.0, 0.5)):
        super().__init__()
        self.x_bound, self.y_bound = x_bound, y_bound
        self.nx = int((x_bound[1] - x_bound[0]) / x_bound[2])
        self.ny = int((y_bound[1] - y_bound[0]) / y_bound[2])
        self.gru = ConvGRUCell(bev_dim, hidden_dim)
        self.input_proj = nn.Conv2d(bev_dim, hidden_dim, 1) if bev_dim != hidden_dim else nn.Identity()

    def warp_bev(self, bev_feat, ego_motion):
        B, C, H, W = bev_feat.shape
        device = bev_feat.device
        xs = torch.linspace(self.x_bound[0]+self.x_bound[2]/2, self.x_bound[1]-self.x_bound[2]/2, self.nx, device=device)
        ys = torch.linspace(self.y_bound[0]+self.y_bound[2]/2, self.y_bound[1]-self.y_bound[2]/2, self.ny, device=device)
        gy, gx = torch.meshgrid(ys, xs, indexing="ij")
        coords = torch.stack([gx, gy, torch.zeros_like(gx), torch.ones_like(gx)], dim=-1).view(-1, 4)
        ego_inv = torch.inverse(ego_motion)
        prev = torch.einsum("bij,pj->bpi", ego_inv, coords)
        nx = (prev[...,0] - self.x_bound[0]) / (self.x_bound[1] - self.x_bound[0]) * 2 - 1
        ny = (prev[...,1] - self.y_bound[0]) / (self.y_bound[1] - self.y_bound[0]) * 2 - 1
        grid = torch.stack([nx, ny], dim=-1).view(B, H, W, 2)
        return F.grid_sample(bev_feat, grid, mode="bilinear", padding_mode="zeros", align_corners=True)

    def forward(self, current_bev, prev_hidden, ego_motion=None):
        current_proj = self.input_proj(current_bev)
        if prev_hidden is None or ego_motion is None:
            return current_proj
        warped = self.warp_bev(prev_hidden, ego_motion)
        return self.gru(current_proj, warped)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BEV ENCODER + OCCUPANCY HEAD
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class ResBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = nn.Sequential(
            nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False), nn.BatchNorm2d(out_c)
        ) if stride != 1 or in_c != out_c else None

    def forward(self, x):
        identity = x if self.downsample is None else self.downsample(x)
        return self.relu(self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))) + identity)


class BEVEncoder(nn.Module):
    def __init__(self, in_channels=64, channels=(64, 128, 256)):
        super().__init__()
        c1, c2, c3 = channels
        self.layer1 = nn.Sequential(ResBlock(in_channels, c1), ResBlock(c1, c1))
        self.layer2 = nn.Sequential(ResBlock(c1, c2, stride=2), ResBlock(c2, c2))
        self.layer3 = nn.Sequential(ResBlock(c2, c3, stride=2), ResBlock(c3, c3))
        self.up3 = nn.Sequential(nn.Conv2d(c3, c2, 1, bias=False), nn.BatchNorm2d(c2),
                                  nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True))
        self.fuse2 = nn.Sequential(nn.Conv2d(c2, c2, 3, padding=1, bias=False),
                                    nn.BatchNorm2d(c2), nn.ReLU(inplace=True))
        self.up2 = nn.Sequential(nn.Conv2d(c2, c1, 1, bias=False), nn.BatchNorm2d(c1),
                                  nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True))
        self.fuse1 = nn.Sequential(nn.Conv2d(c1, c1, 3, padding=1, bias=False),
                                    nn.BatchNorm2d(c1), nn.ReLU(inplace=True))
        self.out_channels = [c1, c2, c3]

    def forward(self, x):
        f1 = self.layer1(x)
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        p2 = self.fuse2(f2 + self.up3(f3))
        p1 = self.fuse1(f1 + self.up2(p2))
        return [p1, p2, f3], p1


class OccupancyHead(nn.Module):
    def __init__(self, in_channels_list=(64, 128, 256), num_classes=1):
        super().__init__()
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(c, c // 2, 3, padding=1, bias=False),
                nn.BatchNorm2d(c // 2), nn.ReLU(inplace=True),
                nn.Conv2d(c // 2, num_classes, 1),
            ) for c in in_channels_list
        ])

    def forward(self, multi_scale_features, target_size=None):
        preds = []
        for feat, head in zip(multi_scale_features, self.heads):
            p = head(feat)
            if target_size is not None:
                p = F.interpolate(p, size=target_size, mode="bilinear", align_corners=True)
            preds.append(p)
        return preds


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FULL MODEL
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class BEVOcc(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        m = cfg["model"]
        self.feat_dim = m["image_feat_dim"]
        self.use_temporal = m["use_temporal"]

        self.backbone = EfficientNetBackbone(out_channels=self.feat_dim, pretrained=True)
        self.depth_net = DepthNet(in_channels=self.feat_dim, depth_channels=m["depth_channels"])
        self.lift_splat = LiftSplat(
            feat_dim=self.feat_dim, depth_channels=m["depth_channels"],
            x_bound=tuple(m["bev_x_bound"]), y_bound=tuple(m["bev_y_bound"]),
            z_bound=tuple(m["bev_z_bound"]), depth_min=m["depth_min"], depth_max=m["depth_max"])

        if self.use_temporal:
            self.temporal = TemporalFusion(
                bev_dim=self.feat_dim, hidden_dim=m["temporal_hidden_dim"],
                x_bound=tuple(m["bev_x_bound"]), y_bound=tuple(m["bev_y_bound"]))

        bev_in = m["temporal_hidden_dim"] if self.use_temporal else self.feat_dim
        self.bev_encoder = BEVEncoder(in_channels=bev_in, channels=tuple(m["bev_encoder_channels"]))
        self.occ_head = OccupancyHead(in_channels_list=tuple(m["bev_encoder_channels"]),
                                       num_classes=m["occupancy_classes"])
        self.nx = int((m["bev_x_bound"][1] - m["bev_x_bound"][0]) / m["bev_x_bound"][2])
        self.ny = int((m["bev_y_bound"][1] - m["bev_y_bound"][0]) / m["bev_y_bound"][2])

    def forward(self, batch, prev_hidden=None):
        images = batch["images"]
        B, N, C_img, H, W = images.shape
        imgs_flat = images.view(B * N, C_img, H, W)
        feats = self.backbone(imgs_flat)
        _, C, fH, fW = feats.shape
        depth, context = self.depth_net(feats)

        feats = feats.view(B, N, C, fH, fW)
        depth = depth.view(B, N, -1, fH, fW)
        context = context.view(B, N, C, fH, fW)

        bev = self.lift_splat(feats, depth, context,
                               batch["intrinsics"], batch["extrinsics"], H, W)

        if self.use_temporal:
            hidden = self.temporal(bev, prev_hidden, batch.get("ego_motion"))
        else:
            hidden = bev

        multi_scale, fused = self.bev_encoder(hidden)
        predictions = self.occ_head(multi_scale, target_size=(self.nx, self.ny))

        return {"predictions": predictions, "hidden": hidden.detach(), "depth": depth}

    @torch.no_grad()
    def predict(self, batch, prev_hidden=None, threshold=0.5):
        self.eval()
        output = self.forward(batch, prev_hidden)
        prob = torch.sigmoid(output["predictions"][0]).squeeze(1)
        return (prob > threshold).float(), prob, output["hidden"]


print(f"✓ Model classes defined")
print(f"  Components: EfficientNetBackbone, DepthNet, LiftSplat, TemporalFusion, BEVEncoder, OccupancyHead, BEVOcc")


## 4. Loss Functions & Metrics

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# DISTANCE-WEIGHTED BCE LOSS (our key novelty)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class DistanceWeightedBCELoss(nn.Module):
    """BCE loss weighted by 1/distance from ego. Near-field errors penalized more."""

    def __init__(self, x_bound=(-50.0, 50.0, 0.5), y_bound=(-50.0, 50.0, 0.5),
                 alpha=1.0, pos_weight=2.5, min_distance=1.0):
        super().__init__()
        self.pos_weight = pos_weight
        nx = int((x_bound[1] - x_bound[0]) / x_bound[2])
        ny = int((y_bound[1] - y_bound[0]) / y_bound[2])
        xs = torch.linspace(x_bound[0] + x_bound[2]/2, x_bound[1] - x_bound[2]/2, nx)
        ys = torch.linspace(y_bound[0] + y_bound[2]/2, y_bound[1] - y_bound[2]/2, ny)
        gy, gx = torch.meshgrid(ys, xs, indexing="ij")
        dist = torch.sqrt(gx**2 + gy**2).clamp(min=min_distance)
        wm = 1.0 / (dist ** alpha)
        wm = wm / wm.mean()
        self.register_buffer("weight_map", wm.unsqueeze(0).unsqueeze(0))

    def forward(self, pred_logits, target):
        pw = torch.tensor([self.pos_weight], device=pred_logits.device)
        bce = F.binary_cross_entropy_with_logits(pred_logits, target, pos_weight=pw, reduction="none")
        w = self.weight_map.to(pred_logits.device)
        if bce.shape[-2:] != w.shape[-2:]:
            w = F.interpolate(w, size=bce.shape[-2:], mode="bilinear", align_corners=True)
        return (bce * w).mean()


class MultiScaleLoss(nn.Module):
    """Deep supervision: loss at each BEV encoder scale."""

    def __init__(self, x_bound=(-50.0, 50.0, 0.5), y_bound=(-50.0, 50.0, 0.5),
                 alpha=1.0, pos_weight=2.5, scale_weights=(1.0, 0.5, 0.25)):
        super().__init__()
        self.scale_weights = scale_weights
        self.loss_fn = DistanceWeightedBCELoss(x_bound, y_bound, alpha, pos_weight)

    def forward(self, predictions, target):
        total = 0.0
        loss_dict = {}
        for i, (pred, w) in enumerate(zip(predictions, self.scale_weights)):
            sl = self.loss_fn(pred, target)
            total = total + w * sl
            loss_dict[f"loss_scale_{i}"] = sl.item()
        loss_dict["total_loss"] = total.item()
        return total, loss_dict


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# EVALUATION METRICS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

class OccupancyMetrics:
    """IoU, distance-weighted error, per-distance-bin breakdown."""

    def __init__(self, x_bound=(-50.0, 50.0, 0.5), y_bound=(-50.0, 50.0, 0.5),
                 distance_bins=(0, 10, 20, 30, 40, 50), threshold=0.5):
        self.threshold = threshold
        self.distance_bins = distance_bins
        nx = int((x_bound[1] - x_bound[0]) / x_bound[2])
        ny = int((y_bound[1] - y_bound[0]) / y_bound[2])
        xs = np.linspace(x_bound[0]+x_bound[2]/2, x_bound[1]-x_bound[2]/2, nx)
        ys = np.linspace(y_bound[0]+y_bound[2]/2, y_bound[1]-y_bound[2]/2, ny)
        gy, gx = np.meshgrid(ys, xs, indexing="ij")
        self.distance_map = np.sqrt(gx**2 + gy**2)
        self.reset()

    def reset(self):
        self.tp = self.fp = self.fn = 0
        self.total_we = self.total_w = 0.0
        self.n = 0
        self.bin_tp = {i: 0 for i in range(len(self.distance_bins)-1)}
        self.bin_fp = {i: 0 for i in range(len(self.distance_bins)-1)}
        self.bin_fn = {i: 0 for i in range(len(self.distance_bins)-1)}

    def update(self, pred_prob, target):
        if isinstance(pred_prob, torch.Tensor): pred_prob = pred_prob.detach().cpu().numpy()
        if isinstance(target, torch.Tensor): target = target.detach().cpu().numpy()
        pb = (pred_prob > self.threshold).astype(np.float32)
        for b in range(pb.shape[0]):
            p, t = pb[b], target[b]
            self.tp += np.sum((p==1)&(t==1))
            self.fp += np.sum((p==1)&(t==0))
            self.fn += np.sum((p==0)&(t==1))
            err = np.abs(p - t)
            d = np.maximum(self.distance_map, 1.0)
            w = 1.0 / d
            self.total_we += np.sum(err * w)
            self.total_w += np.sum(w)
            for i in range(len(self.distance_bins)-1):
                mask = (self.distance_map >= self.distance_bins[i]) & (self.distance_map < self.distance_bins[i+1])
                self.bin_tp[i] += np.sum((p[mask]==1)&(t[mask]==1))
                self.bin_fp[i] += np.sum((p[mask]==1)&(t[mask]==0))
                self.bin_fn[i] += np.sum((p[mask]==0)&(t[mask]==1))
        self.n += pb.shape[0]

    def compute(self):
        r = {}
        r["occupancy_iou"] = self.tp / max(self.tp + self.fp + self.fn, 1)
        r["distance_weighted_error"] = self.total_we / max(self.total_w, 1)
        r["precision"] = self.tp / max(self.tp + self.fp, 1)
        r["recall"] = self.tp / max(self.tp + self.fn, 1)
        for i in range(len(self.distance_bins)-1):
            d0, d1 = self.distance_bins[i], self.distance_bins[i+1]
            r[f"iou_{d0}-{d1}m"] = self.bin_tp[i] / max(self.bin_tp[i]+self.bin_fp[i]+self.bin_fn[i], 1)
        r["n_samples"] = self.n
        return r

    def summary(self):
        r = self.compute()
        lines = ["="*50, "BEV Occupancy Results", "="*50,
                 f"  Occupancy IoU:           {r['occupancy_iou']:.4f}",
                 f"  Distance-Weighted Error: {r['distance_weighted_error']:.4f}",
                 f"  Precision:               {r['precision']:.4f}",
                 f"  Recall:                  {r['recall']:.4f}", "-"*50]
        for i in range(len(self.distance_bins)-1):
            d0, d1 = self.distance_bins[i], self.distance_bins[i+1]
            lines.append(f"    {d0:2d}-{d1:2d}m: {r[f'iou_{d0}-{d1}m']:.4f}")
        return "\n".join(lines)


print("✓ Loss functions and metrics defined")


## 5. nuScenes Dataset Loader

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from pyquaternion import Quaternion
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.splits import create_splits_scenes
from nuscenes.utils.data_classes import LidarPointCloud


class NuScenesBEVDataset(Dataset):
    """nuScenes dataset for BEV occupancy with multi-camera + calibration."""

    CAMERAS = [
        "CAM_FRONT_LEFT", "CAM_FRONT", "CAM_FRONT_RIGHT",
        "CAM_BACK_LEFT", "CAM_BACK", "CAM_BACK_RIGHT",
    ]

    def __init__(self, nusc, split="train", input_size=(256, 704), gt_dir=None):
        self.nusc = nusc
        self.input_size = input_size
        self.gt_dir = gt_dir
        splits = create_splits_scenes()
        scene_names = splits[split] if split in splits else splits["mini_train"] if split == "train" else splits["mini_val"]
        self.samples = []
        for scene in nusc.scene:
            if scene["name"] in scene_names:
                tok = scene["first_sample_token"]
                while tok:
                    self.samples.append(tok)
                    tok = nusc.get("sample", tok)["next"]
        print(f"Dataset [{split}]: {len(self.samples)} samples")

    def __len__(self):
        return len(self.samples)

    def _get_transform(self, translation, rotation):
        T = np.eye(4)
        T[:3, :3] = Quaternion(rotation).rotation_matrix
        T[:3, 3] = translation
        return T

    def __getitem__(self, idx):
        sample = self.nusc.get("sample", self.samples[idx])
        images, intrinsics_list, extrinsics_list = [], [], []

        for cam in self.CAMERAS:
            sd = self.nusc.get("sample_data", sample["data"][cam])
            img = Image.open(os.path.join(self.nusc.dataroot, sd["filename"])).convert("RGB")
            ow, oh = img.size
            img = img.resize((self.input_size[1], self.input_size[0]), Image.BILINEAR)
            img_np = np.array(img).astype(np.float32) / 255.0
            img_np = (img_np - np.array([0.485,0.456,0.406])) / np.array([0.229,0.224,0.225])
            images.append(torch.from_numpy(img_np).permute(2,0,1).float())

            calib = self.nusc.get("calibrated_sensor", sd["calibrated_sensor_token"])
            K = np.array(calib["camera_intrinsic"])
            K[0,:] *= self.input_size[1] / ow
            K[1,:] *= self.input_size[0] / oh
            intrinsics_list.append(torch.from_numpy(K).float())
            extrinsics_list.append(torch.from_numpy(
                self._get_transform(calib["translation"], calib["rotation"])).float())

        # BEV ground truth
        gt_path = os.path.join(self.gt_dir, f"{self.samples[idx]}.npy") if self.gt_dir else None
        if gt_path and os.path.exists(gt_path):
            bev_gt = torch.from_numpy(np.load(gt_path)).float()
        else:
            bev_gt = torch.zeros(200, 200)

        # Ego motion
        ego_motion = torch.eye(4).float()
        if sample["prev"]:
            curr_sd = self.nusc.get("sample_data", sample["data"]["LIDAR_TOP"])
            curr_pose = self.nusc.get("ego_pose", curr_sd["ego_pose_token"])
            T_curr = self._get_transform(curr_pose["translation"], curr_pose["rotation"])
            prev_s = self.nusc.get("sample", sample["prev"])
            prev_sd = self.nusc.get("sample_data", prev_s["data"]["LIDAR_TOP"])
            prev_pose = self.nusc.get("ego_pose", prev_sd["ego_pose_token"])
            T_prev = self._get_transform(prev_pose["translation"], prev_pose["rotation"])
            ego_motion = torch.from_numpy(np.linalg.inv(T_curr) @ T_prev).float()

        return {
            "images": torch.stack(images),
            "intrinsics": torch.stack(intrinsics_list),
            "extrinsics": torch.stack(extrinsics_list),
            "bev_gt": bev_gt,
            "ego_motion": ego_motion,
            "token": self.samples[idx],
        }


def collate_fn(batch):
    return {
        "images": torch.stack([b["images"] for b in batch]),
        "intrinsics": torch.stack([b["intrinsics"] for b in batch]),
        "extrinsics": torch.stack([b["extrinsics"] for b in batch]),
        "bev_gt": torch.stack([b["bev_gt"] for b in batch]),
        "ego_motion": torch.stack([b["ego_motion"] for b in batch]),
        "tokens": [b["token"] for b in batch],
    }


print("✓ Dataset loader defined")


## 6. Generate BEV Ground Truth from LiDAR

In [ ]:
def generate_bev_gt(nusc, output_dir, num_sweeps=10,
                    x_bound=(-50.0, 50.0, 0.5), y_bound=(-50.0, 50.0, 0.5)):
    """Generate BEV occupancy GT from multi-sweep LiDAR."""
    from tqdm import tqdm
    os.makedirs(output_dir, exist_ok=True)

    nx = int((x_bound[1] - x_bound[0]) / x_bound[2])
    ny = int((y_bound[1] - y_bound[0]) / y_bound[2])

    def get_T(trans, rot):
        T = np.eye(4)
        T[:3,:3] = Quaternion(rot).rotation_matrix
        T[:3,3] = trans
        return T

    for sample in tqdm(nusc.sample, desc="Generating GT"):
        out_path = os.path.join(output_dir, f"{sample['token']}.npy")
        if os.path.exists(out_path):
            continue

        sd = nusc.get("sample_data", sample["data"]["LIDAR_TOP"])
        ego_pose = nusc.get("ego_pose", sd["ego_pose_token"])
        T_ego2glob = get_T(ego_pose["translation"], ego_pose["rotation"])
        T_glob2ego = np.linalg.inv(T_ego2glob)
        calib = nusc.get("calibrated_sensor", sd["calibrated_sensor_token"])
        T_sens2ego = get_T(calib["translation"], calib["rotation"])

        all_pts = []
        # Current sweep
        pc = LidarPointCloud.from_file(os.path.join(nusc.dataroot, sd["filename"]))
        pts = pc.points[:3,:].T
        pts_h = np.hstack([pts, np.ones((len(pts),1))])
        all_pts.append((T_sens2ego @ pts_h.T).T[:,:3])

        # Previous sweeps
        cur = sd
        for _ in range(num_sweeps - 1):
            if cur["prev"] == "": break
            cur = nusc.get("sample_data", cur["prev"])
            pc = LidarPointCloud.from_file(os.path.join(nusc.dataroot, cur["filename"]))
            pts = pc.points[:3,:].T
            sc = nusc.get("calibrated_sensor", cur["calibrated_sensor_token"])
            se = nusc.get("ego_pose", cur["ego_pose_token"])
            T_full = T_glob2ego @ get_T(se["translation"], se["rotation"]) @ get_T(sc["translation"], sc["rotation"])
            pts_h = np.hstack([pts, np.ones((len(pts),1))])
            all_pts.append((T_full @ pts_h.T).T[:,:3])

        points = np.concatenate(all_pts, axis=0)
        # Height filter
        mask = (points[:,2] >= -3.0) & (points[:,2] <= 3.0)
        points = points[mask]
        # BEV bounds
        mask = ((points[:,0] >= x_bound[0]) & (points[:,0] < x_bound[1]) &
                (points[:,1] >= y_bound[0]) & (points[:,1] < y_bound[1]))
        points = points[mask]

        grid = np.zeros((ny, nx), dtype=np.float32)
        if len(points) > 0:
            xi = np.clip(((points[:,0] - x_bound[0]) / x_bound[2]).astype(int), 0, nx-1)
            yi = np.clip(((points[:,1] - y_bound[0]) / y_bound[2]).astype(int), 0, ny-1)
            np.add.at(grid, (yi, xi), 1)
            grid = (grid >= 1).astype(np.float32)

        np.save(out_path, grid)

    print(f"✓ Generated GT for {len(nusc.sample)} samples in {output_dir}")


# Run GT generation
DATAROOT = "/content/data/nuscenes"
GT_DIR = "/content/data/nuscenes/bev_gt"

if os.path.exists(os.path.join(DATAROOT, "v1.0-mini")):
    nusc = NuScenes(version="v1.0-mini", dataroot=DATAROOT, verbose=False)
    generate_bev_gt(nusc, GT_DIR, num_sweeps=10)

    # Stats
    import glob
    gts = [np.load(f) for f in glob.glob(f"{GT_DIR}/*.npy")[:50]]
    if gts:
        rates = [g.mean() for g in gts]
        print(f"Occupancy rate: mean={np.mean(rates):.4f}, min={np.min(rates):.4f}, max={np.max(rates):.4f}")
else:
    print("⏳ nuScenes not found. Run the download cell first.")


## 7. Configuration

In [ ]:
CONFIG = dict(
    model=dict(
        backbone="efficientnet-b4",
        image_feat_dim=160,
        depth_channels=112,
        depth_min=1.0,
        depth_max=57.0,
        bev_x_bound=[-50.0, 50.0, 0.5],
        bev_y_bound=[-50.0, 50.0, 0.5],
        bev_z_bound=[-5.0, 3.0, 8.0],
        bev_feat_dim=64,
        bev_encoder_channels=[64, 128, 256],
        use_temporal=False,  # Set True if you have enough VRAM
        temporal_hidden_dim=64,
        occupancy_classes=1,
    ),
    training=dict(
        epochs=20,
        batch_size=2,            # T4: use 2, A100: use 4-6
        num_workers=2,
        lr=2e-4,
        weight_decay=1e-2,
        warmup_epochs=2,
        min_lr=1e-6,
        grad_clip=5.0,
        accumulation_steps=4,    # effective batch = 2 * 4 = 8
        distance_weight_alpha=1.0,
        bce_pos_weight=2.5,
        use_amp=True,
    ),
)

print("✓ Config ready")
print(f"  Batch size: {CONFIG['training']['batch_size']} × {CONFIG['training']['accumulation_steps']} accum = {CONFIG['training']['batch_size'] * CONFIG['training']['accumulation_steps']} effective")
print(f"  Epochs: {CONFIG['training']['epochs']}")
print(f"  BEV grid: 200×200 (0.5m resolution, 100m×100m)")


## 8. Train!

In [ ]:
import time
from torch.cuda.amp import GradScaler, autocast

def train():
    DATAROOT = "/content/data/nuscenes"
    GT_DIR = "/content/data/nuscenes/bev_gt"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cfg = CONFIG

    # Dataset
    nusc = NuScenes(version="v1.0-mini", dataroot=DATAROOT, verbose=False)
    train_ds = NuScenesBEVDataset(nusc, split="train", gt_dir=GT_DIR)
    val_ds = NuScenesBEVDataset(nusc, split="val", gt_dir=GT_DIR)
    train_loader = DataLoader(train_ds, batch_size=cfg["training"]["batch_size"],
                               shuffle=True, num_workers=cfg["training"]["num_workers"],
                               collate_fn=collate_fn, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg["training"]["batch_size"],
                             shuffle=False, num_workers=cfg["training"]["num_workers"],
                             collate_fn=collate_fn, pin_memory=True)

    # Model
    model = BEVOcc(cfg).to(device)
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: {params/1e6:.1f}M trainable parameters")

    # Loss
    m = cfg["model"]
    criterion = MultiScaleLoss(
        x_bound=tuple(m["bev_x_bound"]), y_bound=tuple(m["bev_y_bound"]),
        alpha=cfg["training"]["distance_weight_alpha"],
        pos_weight=cfg["training"]["bce_pos_weight"])

    # Optimizer
    backbone_p = [p for n,p in model.named_parameters() if "backbone" in n]
    other_p = [p for n,p in model.named_parameters() if "backbone" not in n]
    optimizer = torch.optim.AdamW([
        {"params": backbone_p, "lr": cfg["training"]["lr"] * 0.1},
        {"params": other_p, "lr": cfg["training"]["lr"]},
    ], weight_decay=cfg["training"]["weight_decay"])

    total_steps = cfg["training"]["epochs"] * len(train_loader) // cfg["training"]["accumulation_steps"]
    warmup_steps = cfg["training"]["warmup_epochs"] * len(train_loader) // cfg["training"]["accumulation_steps"]

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return max(cfg["training"]["min_lr"] / cfg["training"]["lr"],
                   0.5 * (1 + math.cos(math.pi * progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = GradScaler(enabled=cfg["training"]["use_amp"])

    best_iou = 0.0
    accum = cfg["training"]["accumulation_steps"]
    history = {"train_loss": [], "val_loss": [], "val_iou": [], "val_dwe": []}

    for epoch in range(cfg["training"]["epochs"]):
        # ── Train ──
        model.train()
        epoch_loss = 0.0
        optimizer.zero_grad()
        t0 = time.time()

        for i, batch in enumerate(train_loader):
            imgs = batch["images"].to(device)
            intr = batch["intrinsics"].to(device)
            extr = batch["extrinsics"].to(device)
            gt = batch["bev_gt"].to(device).unsqueeze(1)

            with autocast(enabled=cfg["training"]["use_amp"]):
                out = model({"images": imgs, "intrinsics": intr, "extrinsics": extr})
                loss, ld = criterion(out["predictions"], gt)
                loss = loss / accum

            scaler.scale(loss).backward()

            if (i + 1) % accum == 0:
                if cfg["training"]["grad_clip"] > 0:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), cfg["training"]["grad_clip"])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()

            epoch_loss += ld["total_loss"]

            if i % 20 == 0:
                print(f"  [{i}/{len(train_loader)}] loss={ld['total_loss']:.4f}", end="\r")

        avg_train_loss = epoch_loss / len(train_loader)

        # ── Validate ──
        model.eval()
        metrics = OccupancyMetrics()
        val_loss_sum = 0.0

        with torch.no_grad():
            for batch in val_loader:
                imgs = batch["images"].to(device)
                intr = batch["intrinsics"].to(device)
                extr = batch["extrinsics"].to(device)
                gt = batch["bev_gt"].to(device)

                with autocast(enabled=cfg["training"]["use_amp"]):
                    out = model({"images": imgs, "intrinsics": intr, "extrinsics": extr})
                    loss, _ = criterion(out["predictions"], gt.unsqueeze(1))

                val_loss_sum += loss.item()
                pred_prob = torch.sigmoid(out["predictions"][0].squeeze(1))
                metrics.update(pred_prob, gt)

        val_loss = val_loss_sum / max(len(val_loader), 1)
        results = metrics.compute()
        elapsed = time.time() - t0

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(val_loss)
        history["val_iou"].append(results["occupancy_iou"])
        history["val_dwe"].append(results["distance_weighted_error"])

        is_best = results["occupancy_iou"] > best_iou
        if is_best:
            best_iou = results["occupancy_iou"]
            torch.save({"epoch": epoch, "model": model.state_dict(),
                        "best_iou": best_iou, "results": results},
                       "/content/best_model.pth")

        star = " ★" if is_best else ""
        print(f"Epoch {epoch:2d} ({elapsed:.0f}s) | "
              f"Train: {avg_train_loss:.4f} | "
              f"Val: {val_loss:.4f} | "
              f"IoU: {results['occupancy_iou']:.4f} | "
              f"DWE: {results['distance_weighted_error']:.4f}{star}")

    print(f"\n{'='*60}")
    print(f"Training complete! Best IoU: {best_iou:.4f}")
    print(f"{'='*60}")

    # Save final
    torch.save({"epoch": cfg["training"]["epochs"]-1, "model": model.state_dict(),
                "best_iou": best_iou, "history": history},
               "/content/final_model.pth")

    return model, history, metrics


model, history, final_metrics = train()


## 9. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history["train_loss"], label="Train", linewidth=2)
axes[0].plot(history["val_loss"], label="Val", linewidth=2)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Loss Curves"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history["val_iou"], "g-o", linewidth=2, markersize=4)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("IoU")
axes[1].set_title("Occupancy IoU (↑)"); axes[1].grid(True, alpha=0.3)

axes[2].plot(history["val_dwe"], "r-o", linewidth=2, markersize=4)
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("DWE")
axes[2].set_title("Distance-Weighted Error (↓)"); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/content/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: /content/training_curves.png")


## 10. Final Evaluation

In [ ]:
# Load best model
ckpt = torch.load("/content/best_model.pth")
print(f"Best model from epoch {ckpt['epoch']}")
print(f"\nFinal Metrics:")
print(final_metrics.summary())

# Per-distance breakdown
results = final_metrics.compute()
print(f"\n{'='*40}")
print(f"SUBMISSION METRICS:")
print(f"{'='*40}")
print(f"  Occupancy IoU:           {results['occupancy_iou']:.4f}")
print(f"  Distance-Weighted Error: {results['distance_weighted_error']:.4f}")
print(f"  Precision:               {results['precision']:.4f}")
print(f"  Recall:                  {results['recall']:.4f}")
for i in range(len(final_metrics.distance_bins)-1):
    d0, d1 = final_metrics.distance_bins[i], final_metrics.distance_bins[i+1]
    print(f"  IoU {d0:2d}-{d1:2d}m:             {results[f'iou_{d0}-{d1}m']:.4f}")


## 11. Visualize Predictions

In [ ]:
import matplotlib
from matplotlib.colors import LinearSegmentedColormap

def visualize_sample(model, dataset, idx, device):
    """Visualize a single sample: cameras + BEV prediction + GT."""
    model.eval()
    sample = dataset[idx]

    batch = {
        "images": sample["images"].unsqueeze(0).to(device),
        "intrinsics": sample["intrinsics"].unsqueeze(0).to(device),
        "extrinsics": sample["extrinsics"].unsqueeze(0).to(device),
    }

    with torch.no_grad(), autocast(enabled=True):
        out = model(batch)

    pred = torch.sigmoid(out["predictions"][0]).squeeze().cpu().numpy()
    gt = sample["bev_gt"].numpy()

    colors = ["#1a1a2e", "#16213e", "#0f3460", "#e94560", "#ff6b6b"]
    cmap = LinearSegmentedColormap.from_list("occ", colors)

    fig = plt.figure(figsize=(22, 14))

    # Camera images
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    cam_names = NuScenesBEVDataset.CAMERAS

    for i in range(6):
        ax = fig.add_subplot(3, 4, i + 1 + (2 if i >= 3 else 0))
        img = sample["images"][i].permute(1,2,0).numpy() * std + mean
        ax.imshow(np.clip(img, 0, 1))
        ax.set_title(cam_names[i].replace("CAM_",""), fontsize=10)
        ax.axis("off")

    # BEV prediction
    ax = fig.add_subplot(3, 4, 4)
    ax.imshow(pred, cmap=cmap, vmin=0, vmax=1, origin="lower")
    ax.plot(100, 100, "w^", markersize=10)
    ax.set_title("BEV Prediction", fontsize=12, fontweight="bold")
    ax.axis("off")

    # BEV GT
    ax = fig.add_subplot(3, 4, 8)
    ax.imshow(gt, cmap=cmap, vmin=0, vmax=1, origin="lower")
    ax.plot(100, 100, "w^", markersize=10)
    ax.set_title("Ground Truth (LiDAR)", fontsize=12, fontweight="bold")
    ax.axis("off")

    # Error map
    ax = fig.add_subplot(3, 4, 12)
    ax.imshow(np.abs(pred - gt), cmap="hot", vmin=0, vmax=1, origin="lower")
    ax.plot(100, 100, "w^", markersize=10)
    ax.set_title("Error Map", fontsize=12, fontweight="bold")
    ax.axis("off")

    plt.suptitle(f"BEV-Occ Prediction — Sample {idx}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    return fig


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATAROOT = "/content/data/nuscenes"
GT_DIR = "/content/data/nuscenes/bev_gt"

nusc = NuScenes(version="v1.0-mini", dataroot=DATAROOT, verbose=False)
val_ds = NuScenesBEVDataset(nusc, split="val", gt_dir=GT_DIR)

# Visualize 5 random samples
import random
for idx in random.sample(range(len(val_ds)), min(5, len(val_ds))):
    fig = visualize_sample(model, val_ds, idx, device)
    fig.savefig(f"/content/vis_sample_{idx}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

print("\n✓ Visualizations saved to /content/vis_sample_*.png")


## 12. Download Results

In [ ]:
# Package everything for download
import shutil

os.makedirs("/content/submission", exist_ok=True)
shutil.copy("/content/best_model.pth", "/content/submission/")
shutil.copy("/content/training_curves.png", "/content/submission/")

# Copy visualizations
import glob
for f in glob.glob("/content/vis_sample_*.png"):
    shutil.copy(f, "/content/submission/")

# Create results summary
results = final_metrics.compute()
with open("/content/submission/results.txt", "w") as f:
    f.write("BEV-Occ Results\n")
    f.write("="*50 + "\n")
    f.write(f"Occupancy IoU:           {results['occupancy_iou']:.4f}\n")
    f.write(f"Distance-Weighted Error: {results['distance_weighted_error']:.4f}\n")
    f.write(f"Precision:               {results['precision']:.4f}\n")
    f.write(f"Recall:                  {results['recall']:.4f}\n")

# Zip for easy download
shutil.make_archive("/content/bevocc_submission", "zip", "/content/submission")

print("✓ All artifacts packaged!")
print()
print("Download: /content/bevocc_submission.zip")
print()
print("Contents:")
for f in sorted(os.listdir("/content/submission")):
    size = os.path.getsize(f"/content/submission/{f}") / 1024
    print(f"  {f} ({size:.0f} KB)")

# Auto-download in Colab
try:
    from google.colab import files
    files.download("/content/bevocc_submission.zip")
except:
    print("\nManual download: click the file icon → /content/bevocc_submission.zip")


## 🎯 Done!

**What you have now:**
1. `best_model.pth` — trained model checkpoint
2. `training_curves.png` — loss/IoU/DWE plots for the PPT
3. `vis_sample_*.png` — prediction visualizations for the PPT
4. `results.txt` — final metrics for submission

**Next steps:**
1. Push the GitHub repo code
2. Fill in the hackathon PPT template with these results
3. Link the GitHub repo in the PPT
4. Submit!
